# Experiment 2 : R Programming Lab
## Lab 3: Control Flow for Data Cleaning & Lab 4: Advanced Missing Data Handling

**Name:** Sheshank Singh  
**Roll Number:** 23102A0017  
**Subject:** R Programming  


## Problem Statement

This notebook implements the two practicals specified in Experiment 2:

- **Lab 3 — Control Flow for Data Cleaning:** clean resting blood pressure values, handle errors safely, compare loop-based and vectorized cleaning, and validate the cleaned data.
- **Lab 4 — Advanced Missing Data Handling:** identify `NA`, `NULL`, `NaN`, blank strings, and impossible values; apply appropriate treatment; visualize missingness; and validate the cleaned data.

## Learning Objectives

- Use `if-else`, loops, custom functions, and `tryCatch()` in R.
- Compare iterative and vectorized data cleaning.
- Distinguish between `NA`, `NULL`, `NaN`, and blank strings.
- Apply median imputation and categorical missing-value treatment.
- Validate and export cleaned datasets and summaries.

## 1. Setup and Package Installation

The experiment uses base R for the main cleaning workflows. `naniar` is used for missingness summaries/visualization and `skimr` for dataset validation in Lab 4.

In [20]:
required_packages <- c("naniar", "skimr")
missing_packages <- required_packages[
  !vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)
]
if (length(missing_packages) > 0) {
  install.packages(missing_packages, repos = "https://cloud.r-project.org")
}
library(naniar)
library(skimr)

cat("R version:", R.version.string, "\n")
cat("naniar version:", as.character(packageVersion("naniar")), "\n")
cat("skimr version:", as.character(packageVersion("skimr")), "\n")

dir.create("outputs", showWarnings = FALSE, recursive = TRUE)

R version: R version 4.6.1 (2026-06-24) 
naniar version: 1.1.0 
skimr version: 2.2.2 


# Lab 3 — Control Flow for Data Cleaning

## 2. Load the UCI Heart Disease Dataset

The Cleveland processed dataset is used as the source. The assignment focuses on `trestbps` (resting blood pressure) and `chol` (serum cholesterol). The source file has no header, so column names are assigned according to the UCI processed-data format.

In [2]:
heart_url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
heart_col_names <- c(
  "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
  "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"
)

heart_raw <- tryCatch(
  read.csv(
    heart_url, header = FALSE, na.strings = c("?", ""),
    col.names = heart_col_names, stringsAsFactors = FALSE
  ),
  error = function(e) {
    stop(
      paste0(
        "Could not download the UCI Heart Disease dataset. Check internet access or ",
        "upload the processed.cleveland.data file. Original error: ", conditionMessage(e)
      )
    )
  }
)

heart_data <- heart_raw
heart_data$trestbps <- suppressWarnings(as.numeric(heart_data$trestbps))
heart_data$chol <- suppressWarnings(as.numeric(heart_data$chol))

cat("Rows:", nrow(heart_data), "\n")
cat("Columns:", ncol(heart_data), "\n")
str(heart_data)
head(heart_data)

Rows: 303 
Columns: 14 
'data.frame':	303 obs. of  14 variables:
 $ age     : num  63 67 67 37 41 56 62 57 63 53 ...
 $ sex     : num  1 1 1 1 0 1 0 0 1 1 ...
 $ cp      : num  1 4 4 3 2 2 4 4 4 4 ...
 $ trestbps: num  145 160 120 130 130 120 140 120 130 140 ...
 $ chol    : num  233 286 229 250 204 236 268 354 254 203 ...
 $ fbs     : num  1 0 0 0 0 0 0 0 0 1 ...
 $ restecg : num  2 2 2 0 2 0 2 0 2 2 ...
 $ thalach : num  150 108 129 187 172 178 160 163 147 155 ...
 $ exang   : num  0 1 1 0 0 0 0 1 0 1 ...
 $ oldpeak : num  2.3 1.5 2.6 3.5 1.4 0.8 3.6 0.6 1.4 3.1 ...
 $ slope   : num  3 2 2 3 1 1 3 1 2 3 ...
 $ ca      : num  0 3 2 0 0 0 2 0 1 0 ...
 $ thal    : num  6 3 7 3 3 3 3 3 7 7 ...
 $ num     : int  0 2 1 0 0 0 3 0 2 1 ...


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,63,1,1,145,233,1,2,150,0,2.3,3,0,6,0
2,67,1,4,160,286,0,2,108,1,1.5,2,3,3,2
3,67,1,4,120,229,0,2,129,1,2.6,2,2,7,1
4,37,1,3,130,250,0,0,187,0,3.5,3,0,3,0
5,41,0,2,130,204,0,2,172,0,1.4,1,0,3,0
6,56,1,2,120,236,0,0,178,0,0.8,1,0,3,0


## 3. Introduce Simulated Data-Entry Problems

To follow the assignment instructions, this section deliberately introduces a small number of negative BP values, missing BP values, and extreme readings above 300 mmHg. The original source data remains available in `heart_raw`.

In [3]:
heart_data$trestbps_simulated <- heart_data$trestbps
n <- nrow(heart_data)
if (n < 10) stop("The loaded dataset contains too few rows for the required demonstration.")

# Use fixed row positions so the demonstration is reproducible.
heart_data$trestbps_simulated[1] <- -12
heart_data$trestbps_simulated[2] <- NA_real_
heart_data$trestbps_simulated[3] <- 325
heart_data$trestbps_simulated[4] <- 280

cat("Simulated BP problems introduced:\n")
print(heart_data[1:4, c("trestbps", "trestbps_simulated", "chol")])

Simulated BP problems introduced:
  trestbps trestbps_simulated chol
1      145                -12  233
2      160                 NA  286
3      120                325  229
4      130                280  250


## 4. Create a BP-Cleaning Function Using `if-else`

Rules required by the problem statement:

- Negative BP values become `NA`.
- BP values greater than 250 mmHg are capped at 250.
- Valid values remain unchanged.
- Missing values remain missing.

In [4]:
clean_bp_value <- function(bp) {
  if (is.na(bp)) {
    return(NA_real_)
  } else if (bp < 0) {
    return(NA_real_)
  } else if (bp > 250) {
    return(250)
  } else {
    return(as.numeric(bp))
  }
}

# Demonstrate the function on representative values.
test_bp <- c(-12, NA, 120, 250, 280, 325)
data.frame(
  Input_BP = test_bp,
  Cleaned_BP = vapply(test_bp, clean_bp_value, numeric(1))
)

Input_BP,Cleaned_BP
<dbl>,<dbl>
-12,NA
NA,NA
120,120
250,250
280,250
325,250


## 5. Apply the Cleaning Function to the Dataset

A loop applies the custom function to each simulated BP value. The cleaned values are stored in `trestbps_clean`.

In [5]:
heart_data$trestbps_clean <- vapply(
  heart_data$trestbps_simulated,
  clean_bp_value,
  numeric(1)
)

head(heart_data[, c("age", "trestbps_simulated", "trestbps_clean", "chol")], 10)

,age,trestbps_simulated,trestbps_clean,chol
,<dbl>,<dbl>,<dbl>,<dbl>
1,63,-12,NA,233
2,67,NA,NA,286
3,67,325,250,229
4,37,280,250,250
5,41,130,130,204
6,56,120,120,236
7,62,140,140,268
8,57,120,120,354
9,63,130,130,254


## 6. Error Handling with `tryCatch()`

The following functions safely calculate the mean BP while ignoring missing values and calculate the ratio `chol / trestbps`. Invalid denominators (zero, missing, or negative) return `NA` and an informative warning instead of terminating the program.

In [6]:
safe_mean_bp <- function(x) {
  tryCatch({
    valid <- x[!is.na(x)]
    if (length(valid) == 0) stop("Mean BP cannot be calculated: no non-missing values.")
    mean(valid)
  }, error = function(e) {
    message("ERROR calculating mean BP: ", conditionMessage(e))
    NA_real_
  })
}

safe_chol_bp_ratio <- function(chol, bp) {
  tryCatch({
    if (is.na(chol) || is.na(bp)) stop("Cholesterol or BP is missing.")
    if (!is.finite(chol) || !is.finite(bp)) stop("Cholesterol or BP is not finite.")
    if (bp <= 0) stop("BP denominator must be greater than zero.")
    chol / bp
  }, error = function(e) {
    message("WARNING calculating chol/BP ratio: ", conditionMessage(e))
    NA_real_
  })
}

mean_clean_bp <- safe_mean_bp(heart_data$trestbps_clean)
cat("Mean cleaned BP:", mean_clean_bp, "\n")

# Demonstrate valid and invalid cases.
ratio_examples <- data.frame(
  chol = c(240, 200, 190, 180),
  bp = c(120, 0, NA, -5)
)
ratio_examples$chol_bp_ratio <- mapply(
  safe_chol_bp_ratio, ratio_examples$chol, ratio_examples$bp
)
ratio_examples

Mean cleaned BP: 132.3821 


WARNING calculating chol/BP ratio: BP denominator must be greater than zero.

WARNING calculating chol/BP ratio: Cholesterol or BP is missing.

WARNING calculating chol/BP ratio: BP denominator must be greater than zero.



chol,bp,chol_bp_ratio
<dbl>,<dbl>,<dbl>
240,120,2
200,0,NA
190,NA,NA
180,-5,NA


## 7. Compare Loop-Based and Vectorized Cleaning

Both approaches apply the same rules to the simulated BP vector. `system.time()` measures elapsed execution time. Because this dataset is small, timings can be close to zero and may vary between runs.

In [7]:
bp_input <- heart_data$trestbps_simulated

clean_bp_loop <- function(x) {
  out <- x
  for (i in seq_along(x)) {
    if (is.na(x[i])) {
      out[i] <- NA_real_
    } else if (x[i] < 0) {
      out[i] <- NA_real_
    } else if (x[i] > 250) {
      out[i] <- 250
    } else {
      out[i] <- x[i]
    }
  }
  out
}

clean_bp_vectorized <- function(x) {
  out <- x
  out[!is.na(out) & out < 0] <- NA_real_
  out[!is.na(out) & out > 250] <- 250
  out
}

loop_timing <- system.time(loop_result <- clean_bp_loop(bp_input))
vector_timing <- system.time(vector_result <- clean_bp_vectorized(bp_input))

timing_comparison <- data.frame(
  Method = c("For loop", "Vectorized operations"),
  User_Seconds = c(loop_timing[["user.self"]], vector_timing[["user.self"]]),
  System_Seconds = c(loop_timing[["sys.self"]], vector_timing[["sys.self"]]),
  Elapsed_Seconds = c(loop_timing[["elapsed"]], vector_timing[["elapsed"]])
)
timing_comparison

cat("Identical results:", isTRUE(all.equal(loop_result, vector_result)), "\n")

Method,User_Seconds,System_Seconds,Elapsed_Seconds
<chr>,<dbl>,<dbl>,<dbl>
For loop,0.016,0,0.019
Vectorized operations,0.000,0,0.001


Identical results: TRUE 


## 8. Validate the Cleaned BP Data

The validation reports missing values, minimum, maximum, mean, and median. It also checks whether any negative values or values above 250 remain among non-missing cleaned observations.

In [8]:
bp_clean <- heart_data$trestbps_clean
valid_bp <- bp_clean[!is.na(bp_clean)]

bp_validation <- data.frame(
  Measure = c(
    "Total observations",
    "Missing BP values",
    "Minimum BP",
    "Maximum BP",
    "Mean BP",
    "Median BP",
    "Negative values remaining",
    "Values above 250 remaining"
  ),
  Value = c(
    length(bp_clean),
    sum(is.na(bp_clean)),
    if (length(valid_bp)) min(valid_bp) else NA,
    if (length(valid_bp)) max(valid_bp) else NA,
    if (length(valid_bp)) mean(valid_bp) else NA,
    if (length(valid_bp)) median(valid_bp) else NA,
    sum(valid_bp < 0),
    sum(valid_bp > 250)
  )
)
bp_validation

stopifnot(all(valid_bp >= 0), all(valid_bp <= 250))
cat("Validation passed: no negative or >250 values remain among non-missing cleaned BP values.\n")

Measure,Value
<chr>,<dbl>
Total observations,303.0000
Missing BP values,2.0000
Minimum BP,94.0000
Maximum BP,250.0000
Mean BP,132.3821
Median BP,130.0000
Negative values remaining,0.0000
Values above 250 remaining,0.0000


Validation passed: no negative or >250 values remain among non-missing cleaned BP values.


## 9. Export Lab 3 Deliverables

The cleaned data, timing comparison, and BP validation summary are saved in `outputs/`.

In [9]:
cleaned_heart_data <- heart_data
write.csv(cleaned_heart_data, "outputs/cleaned_heart_data.csv", row.names = FALSE)
write.csv(timing_comparison, "outputs/loop_vs_vectorized_timing.csv", row.names = FALSE)
write.csv(bp_validation, "outputs/bp_validation_summary.csv", row.names = FALSE)

writeLines(
  c(
    "Lab 3 conclusion",
    paste("Loop and vectorized outputs identical:", isTRUE(all.equal(loop_result, vector_result))),
    paste("Loop elapsed seconds:", loop_timing[["elapsed"]]),
    paste("Vectorized elapsed seconds:", vector_timing[["elapsed"]]),
    "Vectorized operations are generally concise and often faster for larger vectors; measured timings on this small dataset may not be stable."
  ),
  "outputs/lab3_conclusion.txt"
)

list.files("outputs")

[1] "bp_validation_summary.csv"     "cleaned_heart_data.csv"       
[3] "lab3_conclusion.txt"           "loop_vs_vectorized_timing.csv"

# Lab 4 — Advanced Missing Data Handling

## 10. Load the UCI Adult / Census Income Dataset

The UCI Adult dataset includes demographic and employment variables such as age, work class, education, occupation, hours worked, and income. The dataset is loaded from the UCI repository.

In [10]:
adult_url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
adult_col_names <- c(
  "age", "workclass", "fnlwgt", "education", "education_num",
  "marital_status", "occupation", "relationship", "race", "sex",
  "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"
)

adult_raw <- tryCatch(
  read.csv(
    adult_url, header = FALSE, col.names = adult_col_names,
    na.strings = character(0), strip.white = TRUE,
    stringsAsFactors = FALSE, comment.char = ""
  ),
  error = function(e) {
    stop(
      paste0(
        "Could not download the UCI Adult dataset. Check internet access or upload adult.data. Original error: ",
        conditionMessage(e)
      )
    )
  }
)

# UCI Adult uses '?' to denote missing categorical values.
adult_raw[] <- lapply(adult_raw, function(x) {
  if (is.character(x)) trimws(x) else x
})
cat("Rows:", nrow(adult_raw), "\n")
cat("Columns:", ncol(adult_raw), "\n")
str(adult_raw)
head(adult_raw)

Rows: 32561 
Columns: 15 
'data.frame':	32561 obs. of  15 variables:
 $ age           : int  39 50 38 53 28 37 49 52 31 42 ...
 $ workclass     : chr  "State-gov" "Self-emp-not-inc" "Private" "Private" ...
 $ fnlwgt        : int  77516 83311 215646 234721 338409 284582 160187 209642 45781 159449 ...
 $ education     : chr  "Bachelors" "Bachelors" "HS-grad" "11th" ...
 $ education_num : int  13 13 9 7 13 14 5 9 14 13 ...
 $ marital_status: chr  "Never-married" "Married-civ-spouse" "Divorced" "Married-civ-spouse" ...
 $ occupation    : chr  "Adm-clerical" "Exec-managerial" "Handlers-cleaners" "Handlers-cleaners" ...
 $ relationship  : chr  "Not-in-family" "Husband" "Not-in-family" "Husband" ...
 $ race          : chr  "White" "White" "White" "Black" ...
 $ sex           : chr  "Male" "Male" "Male" "Male" ...
 $ capital_gain  : int  2174 0 0 0 0 0 0 0 14084 5178 ...
 $ capital_loss  : int  0 0 0 0 0 0 0 0 0 0 ...
 $ hours_per_week: int  40 13 40 40 40 40 16 45 50 40 ...
 $ native_country:

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
,<int>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<chr>
1,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
2,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
3,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
4,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
5,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
6,37,Private,284582,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States,<=50K


## 11. Introduce Different Missing and Invalid-Data Scenarios

The assignment asks for deliberate examples of `NA`, blank strings, `NaN`, and impossible numeric values such as `age = 999`. These are introduced into a working copy for demonstration; the downloaded source is retained in `adult_raw`.

In [11]:
adult_data <- adult_raw
n_adult <- nrow(adult_data)
if (n_adult < 10) stop("The Adult dataset contains too few rows for the required demonstration.")

# Convert '?' placeholders to blank strings before simulating the required cases.
adult_data[] <- lapply(adult_data, function(x) {
  if (is.character(x)) {
    x[x == "?"] <- ""
  }
  x
})

adult_data$age <- suppressWarnings(as.numeric(adult_data$age))
adult_data$fnlwgt <- suppressWarnings(as.numeric(adult_data$fnlwgt))
adult_data$education_num <- suppressWarnings(as.numeric(adult_data$education_num))
adult_data$capital_gain <- suppressWarnings(as.numeric(adult_data$capital_gain))
adult_data$capital_loss <- suppressWarnings(as.numeric(adult_data$capital_loss))
adult_data$hours_per_week <- suppressWarnings(as.numeric(adult_data$hours_per_week))

adult_data$age[1] <- NA_real_
adult_data$workclass[2] <- ""
adult_data$hours_per_week[3] <- NaN
adult_data$age[4] <- 999

cat("Rows showing deliberately introduced examples:\n")
adult_data[1:4, c("age", "workclass", "hours_per_week", "income")]

Rows showing deliberately introduced examples:


,age,workclass,hours_per_week,income
,<dbl>,<chr>,<dbl>,<chr>
1,NA,State-gov,40,<=50K
2,50,,13,<=50K
3,38,Private,NaN,<=50K
4,999,Private,40,<=50K


## 12. Identify `NA`, `NaN`, `NULL`, Blank Strings, and Impossible Values

In R, `NaN` is also treated as missing by `is.na()`, while `is.nan()` specifically identifies NaN values. `NULL` represents the absence of an object/value and is not a normal cell value in a data frame. Blank strings (`""`) are character values and must be checked separately.

In [12]:
# Demonstrate the distinctions directly.
missing_type_demo <- list(
  is_na_on_NA = is.na(NA),
  is_na_on_NaN = is.na(NaN),
  is_nan_on_NaN = is.nan(NaN),
  is_null_on_NULL = is.null(NULL),
  blank_string_equals_empty = ("" == "")
)
missing_type_demo

# Count missingness and blanks in the working data.
count_na <- sum(is.na(adult_data$age))
count_nan_hours <- sum(is.nan(adult_data$hours_per_week))
count_blank_workclass <- sum(adult_data$workclass == "", na.rm = TRUE)
count_impossible_age <- sum(adult_data$age == 999, na.rm = TRUE)

data.frame(
  Check = c("NA in age", "NaN in hours_per_week", "Blank workclass", "age == 999"),
  Count = c(count_na, count_nan_hours, count_blank_workclass, count_impossible_age)
)

# NULL is not a cell-level missing marker in a data frame.
null_example <- NULL
is.null(null_example)

$is_na_on_NA
[1] TRUE

$is_na_on_NaN
[1] TRUE

$is_nan_on_NaN
[1] TRUE

$is_null_on_NULL
[1] TRUE

$blank_string_equals_empty
[1] TRUE

Check,Count
<chr>,<int>
NA in age,1
NaN in hours_per_week,1
Blank workclass,1837
age == 999,1


[1] TRUE

## 13. Missing-Value Summary Before Cleaning

Use `naniar::miss_var_summary()` to summarize missingness by variable. Blank strings are counted separately because they are not automatically treated as missing by `is.na()`.

In [13]:
missing_summary_before <- naniar::miss_var_summary(adult_data)
missing_summary_before

blank_summary_before <- data.frame(
  Variable = names(adult_data),
  Blank_Strings = vapply(adult_data, function(x) {
    if (is.character(x)) sum(x == "", na.rm = TRUE) else 0L
  }, integer(1))
)
blank_summary_before

missing_count_before <- sum(is.na(adult_data))
missing_percent_before <- 100 * missing_count_before / (nrow(adult_data) * ncol(adult_data))
cat("Missing cells before treatment:", missing_count_before, "\n")
cat("Missing-cell percentage before treatment:", round(missing_percent_before, 3), "%\n")

variable,n_miss,pct_miss
<chr>,<int>,<num>
age,1,0.003[4m0[24m[4m7[24m
hours_per_week,1,0.003[4m0[24m[4m7[24m
workclass,0,0
fnlwgt,0,0
education,0,0
education_num,0,0
marital_status,0,0
occupation,0,0
relationship,0,0


,Variable,Blank_Strings
,<chr>,<int>
age,age,0
workclass,workclass,1837
fnlwgt,fnlwgt,0
education,education,0
education_num,education_num,0
marital_status,marital_status,0
occupation,occupation,1843
relationship,relationship,0
race,race,0


Missing cells before treatment: 2 
Missing-cell percentage before treatment: 0 %


## 14. Apply Missing-Data Treatment

The treatment strategy follows the assignment:

- Convert impossible `age = 999` values to `NA`.
- Convert blank categorical values to `"Unknown"`.
- Remove rows containing unrecoverable `NaN` values where appropriate.
- Impute missing values in selected numeric variables using their median.
- Use `complete.cases()` to identify complete and incomplete rows.

For this notebook, rows containing NaN are removed before median imputation. Missing numeric values that remain are imputed using the median of valid observations.

In [15]:
# Replace impossible age values with NA.
adult_data$age[!is.na(adult_data$age) & adult_data$age == 999] <- NA_real_

# Convert blank categorical fields (and UCI '?' placeholders if any remain) to Unknown.
categorical_cols <- names(adult_data)[vapply(adult_data, is.character, logical(1))]
for (nm in categorical_cols) {
  adult_data[[nm]][is.na(adult_data[[nm]]) | trimws(adult_data[[nm]]) == "" | adult_data[[nm]] == "?"] <- "Unknown"
}

# Identify rows with any NaN values and remove them as specified where appropriate.
nan_rows <- apply(adult_data, 1, function(row) FALSE)
# Data-frame columns have mixed types; detect NaN safely in numeric columns.
nan_rows <- rep(FALSE, nrow(adult_data))
for (nm in names(adult_data)) {
  if (is.numeric(adult_data[[nm]])) {
    nan_rows <- nan_rows | is.nan(adult_data[[nm]])
  }
}
rows_removed_nan <- sum(nan_rows)
if (rows_removed_nan > 0) adult_data <- adult_data[!nan_rows, , drop = FALSE]

cat("Rows removed due to NaN:", rows_removed_nan, "\n")
cat("Rows remaining:", nrow(adult_data), "\n")

# Custom median-imputation function.
median_impute <- function(x) {
  if (!is.numeric(x)) stop("median_impute() expects a numeric vector.")
  missing_idx <- is.na(x)
  valid_values <- x[!is.na(x) & is.finite(x)]
  if (length(valid_values) == 0) {
    warning("No valid numeric observations available; missing values cannot be imputed.")
    return(x)
  }
  med <- median(valid_values)
  x[missing_idx] <- med
  x
}

numeric_cols <- names(adult_data)[vapply(adult_data, is.numeric, logical(1))]
for (nm in numeric_cols) {
  adult_data[[nm]] <- median_impute(adult_data[[nm]])
}

# Identify complete and incomplete rows after treatment.
complete_row_flags <- complete.cases(adult_data)
cat("Complete rows after treatment:", sum(complete_row_flags), "\n")
cat("Incomplete rows after treatment:", sum(!complete_row_flags), "\n")

Rows removed due to NaN: 1 
Rows remaining: 32560 
Complete rows after treatment: 32560 
Incomplete rows after treatment: 0 


## 15. Missingness After Cleaning

Recalculate missing-value and blank-string summaries to compare the dataset before and after treatment.

In [16]:
missing_summary_after <- naniar::miss_var_summary(adult_data)
missing_summary_after

blank_summary_after <- data.frame(
  Variable = names(adult_data),
  Blank_Strings = vapply(adult_data, function(x) {
    if (is.character(x)) sum(x == "", na.rm = TRUE) else 0L
  }, integer(1))
)
blank_summary_after

missing_count_after <- sum(is.na(adult_data))
missing_percent_after <- 100 * missing_count_after / (nrow(adult_data) * ncol(adult_data))

missingness_comparison <- data.frame(
  Stage = c("Before treatment", "After treatment"),
  Rows = c(nrow(adult_raw), nrow(adult_data)),
  Missing_Cells = c(missing_count_before, missing_count_after),
  Missing_Percent = c(missing_percent_before, missing_percent_after)
)
missingness_comparison

variable,n_miss,pct_miss
<chr>,<int>,<num>
age,0,0
workclass,0,0
fnlwgt,0,0
education,0,0
education_num,0,0
marital_status,0,0
occupation,0,0
relationship,0,0
race,0,0


,Variable,Blank_Strings
,<chr>,<int>
age,age,0
workclass,workclass,0
fnlwgt,fnlwgt,0
education,education,0
education_num,education_num,0
marital_status,marital_status,0
occupation,occupation,0
relationship,relationship,0
race,race,0


Stage,Rows,Missing_Cells,Missing_Percent
<chr>,<int>,<int>,<dbl>
Before treatment,32561,2,0.0004094878
After treatment,32560,0,0.0000000000


## 16. Validate the Cleaned Adult Dataset

Use `skimr::skim()` for a comprehensive overview. Then verify that impossible age values are gone, blank categorical values have been replaced, and selected numeric variables contain no untreated missing values.

In [18]:
skimr::skim(adult_data)

age_999_remaining <- sum(adult_data$age == 999, na.rm = TRUE)
blank_categorical_remaining <- sum(vapply(adult_data[categorical_cols], function(x) {
  sum(x == "", na.rm = TRUE)
}, numeric(1)))

selected_numeric <- intersect(
  c("age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"),
  names(adult_data)
)
numeric_missing_after <- vapply(adult_data[selected_numeric], function(x) sum(is.na(x)), integer(1))

validation_results <- data.frame(
  Check = c(
    "Impossible age values remaining",
    "Blank categorical values remaining",
    paste0("Missing values in ", selected_numeric)
  ),
  Count = c(
    age_999_remaining,
    blank_categorical_remaining,
    as.numeric(numeric_missing_after)
  )
)
validation_results

stopifnot(age_999_remaining == 0)
stopifnot(blank_categorical_remaining == 0)
stopifnot(all(numeric_missing_after == 0))
cat("Validation passed for impossible age values, blank categorical values, and selected numeric variables.\n")

,skim_type,skim_variable,n_missing,complete_rate,character.min,character.max,character.empty,character.n_unique,character.whitespace,numeric.mean,numeric.sd,numeric.p0,numeric.p25,numeric.p50,numeric.p75,numeric.p100,numeric.hist
,<chr>,<chr>,<int>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,character,workclass,0,1,7,16,0,9,0,NA,NA,NA,NA,NA,NA,NA,NA
2,character,education,0,1,3,12,0,16,0,NA,NA,NA,NA,NA,NA,NA,NA
3,character,marital_status,0,1,7,21,0,7,0,NA,NA,NA,NA,NA,NA,NA,NA
4,character,occupation,0,1,5,17,0,15,0,NA,NA,NA,NA,NA,NA,NA,NA
5,character,relationship,0,1,4,14,0,6,0,NA,NA,NA,NA,NA,NA,NA,NA
6,character,race,0,1,5,18,0,5,0,NA,NA,NA,NA,NA,NA,NA,NA
7,character,sex,0,1,4,6,0,2,0,NA,NA,NA,NA,NA,NA,NA,NA
8,character,native_country,0,1,4,26,0,42,0,NA,NA,NA,NA,NA,NA,NA,NA
9,character,income,0,1,4,5,0,2,0,NA,NA,NA,NA,NA,NA,NA,NA


Check,Count
<chr>,<dbl>
Impossible age values remaining,0
Blank categorical values remaining,0
Missing values in age,0
Missing values in fnlwgt,0
Missing values in education_num,0
Missing values in capital_gain,0
Missing values in capital_loss,0
Missing values in hours_per_week,0


Validation passed for impossible age values, blank categorical values, and selected numeric variables.


## 17. Export Lab 4 Deliverables

The cleaned dataset, before/after missingness summaries, and validation results are exported to the `outputs/` folder.

In [22]:
# Helper function to strip special pillar formatting classes from a data frame
strip_pillar_classes <- function(df) {
  df[] <- lapply(df, function(x) {
    if (inherits(x, "vctrs_vctr")) {
      return(as.vector(x))
    }
    return(x)
  })
  as.data.frame(df)
}

write.csv(adult_data, "outputs/cleaned_adult_data.csv", row.names = FALSE)
write.csv(strip_pillar_classes(missing_summary_before), "outputs/adult_missing_summary_before.csv", row.names = FALSE)
write.csv(strip_pillar_classes(missing_summary_after), "outputs/adult_missing_summary_after.csv", row.names = FALSE)
write.csv(missingness_comparison, "outputs/adult_missingness_comparison.csv", row.names = FALSE)
write.csv(validation_results, "outputs/adult_validation_results.csv", row.names = FALSE)

writeLines(
  c(
    "Lab 4 conclusion",
    paste("Rows in source dataset:", nrow(adult_raw)),
    paste("Rows after treatment:", nrow(adult_data)),
    paste("Missing cells before treatment:", missing_count_before),
    paste("Missing cells after treatment:", missing_count_after),
    paste("Rows removed due to NaN:", rows_removed_nan),
    "Impossible age values were converted to NA before numeric imputation.",
    "Blank categorical values were replaced with Unknown.",
    "Selected numeric variables were median-imputed."
  ),
  "outputs/lab4_conclusion.txt"
)

list.files("outputs")

[1] "adult_missing_summary_after.csv"  "adult_missing_summary_before.csv"
 [3] "adult_missingness_comparison.csv" "adult_validation_results.csv"    
 [5] "bp_validation_summary.csv"        "cleaned_adult_data.csv"          
 [7] "cleaned_heart_data.csv"           "lab3_conclusion.txt"             
 [9] "lab4_conclusion.txt"              "loop_vs_vectorized_timing.csv"

## 18. Final Project Structure

The project outputs will include:

```text
Experiment-02-R-Programming/
├── README.md
├── Experiment_02_R_Programming_Lab3_Lab4.ipynb
└── outputs/
    ├── cleaned_heart_data.csv
    ├── loop_vs_vectorized_timing.csv
    ├── bp_validation_summary.csv
    ├── lab3_conclusion.txt
    ├── cleaned_adult_data.csv
    ├── adult_missing_summary_before.csv
    ├── adult_missing_summary_after.csv
    ├── adult_missingness_comparison.csv
    ├── adult_validation_results.csv
    └── lab4_conclusion.txt
```


## 19. Overall Conclusion

This experiment applies core R programming techniques to two practical data-cleaning problems. Lab 3 demonstrates conditional logic, reusable functions, `tryCatch()` error handling, loops, vectorized operations, timing comparisons, and BP validation. Lab 4 distinguishes `NA`, `NULL`, `NaN`, and blank strings, applies missing-data treatment and median imputation, visualizes missingness, and validates the cleaned dataset.